In [3]:
# Model loading and prediction helper

!pip install -q -U transformers accelerate safetensors sentencepiece rdkit

import json
import re
import time
from typing import Any, Dict, Optional

import torch
from rdkit import Chem
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_ID = "oleh13/ord-retro-qwen25-15b-merged-fp16"

SYSTEM_PROMPT = """
You are an expert Organic Chemist specializing in Retrosynthesis.
Your task is to propose practical alternative forward synthesis routes that make the given Target Molecule.

CRITICAL RULES:
1. Output MUST be valid JSON only.
2. Use standard SMILES strings for all molecules.
3. Generate exactly 1 distinct route option.
4. Each route must contain exactly one reaction step.
5. The only step product_smiles in every route MUST be the target molecule SMILES after canonicalization.
6. Do not output Markdown formatting.
7. Use realistic literature-like reaction conditions.
8. Do not invent evidence reaction IDs if unsupported.

JSON STRUCTURE:
{
  "routes": [
    {
      "route_name": "Short descriptive route name",
      "summary": "Why this route is distinct",
      "steps": [
        {
          "reaction_name": "Named or descriptive reaction",
          "reactants": ["SMILES_A", "SMILES_B"],
          "product_smiles": "SMILES_PRODUCT",
          "stoichiometry": "Example: A 1.0 equiv, B 1.2 equiv, base 2.0 equiv",
          "reagents": "Catalysts, bases, acids, oxidants, additives",
          "solvent": "Solvent or solvent mixture",
          "temperature_celsius": "temperature or range with units",
          "reaction_time": "time or range with units",
          "atmosphere": "air, nitrogen, argon, oxygen-free, etc.",
          "workup_purification": "Quench, extraction, chromatography, crystallization, etc.",
          "expected_yield_percent": "estimated percent or range with percent sign",
          "important_conditions": "Other important operational details",
          "rationale": "Why this single reaction forms the target product",
          "objective_fit": "How this route addresses the selected optimization objective",
          "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
        }
      ],
      "objective_fit": "How this route addresses the selected optimization objective",
      "evidence_reaction_ids": ["reaction_id_1", "reaction_id_2"]
    }
  ],
  "overall_recommendation": "Which returned route is preferred and why"
}
""".strip()


def canonical_smiles(smiles: str) -> Optional[str]:
    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None


def extract_json_object(text: str) -> str:
    text = text.strip()

    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    first = text.find("{")
    last = text.rfind("}")

    if first == -1 or last == -1 or last <= first:
        raise ValueError("No JSON object found in model output.")

    return text[first:last + 1]


def validate_route_json(data: Dict[str, Any], target_smiles: str) -> Dict[str, Any]:
    target_can = canonical_smiles(target_smiles)

    result = {
        "valid": True,
        "issues": [],
    }

    routes = data.get("routes")

    if not isinstance(routes, list):
        result["valid"] = False
        result["issues"].append("Missing or invalid routes list.")
        return result

    if len(routes) != 1:
        result["valid"] = False
        result["issues"].append(f"Expected exactly 1 route, got {len(routes)}.")

    if not routes:
        return result

    steps = routes[0].get("steps")

    if not isinstance(steps, list):
        result["valid"] = False
        result["issues"].append("Missing or invalid steps list.")
        return result

    if len(steps) != 1:
        result["valid"] = False
        result["issues"].append(f"Expected exactly 1 step, got {len(steps)}.")

    if not steps:
        return result

    step = steps[0]

    product = step.get("product_smiles")
    product_can = canonical_smiles(product) if isinstance(product, str) else None

    if target_can and product_can and target_can != product_can:
        result["valid"] = False
        result["issues"].append(
            f"product_smiles mismatch: target={target_can}, output={product_can}"
        )

    required_step_fields = [
        "reaction_name",
        "reactants",
        "product_smiles",
        "stoichiometry",
        "reagents",
        "solvent",
        "temperature_celsius",
        "reaction_time",
        "atmosphere",
        "workup_purification",
        "expected_yield_percent",
        "important_conditions",
        "rationale",
        "objective_fit",
        "evidence_reaction_ids",
    ]

    missing = [field for field in required_step_fields if field not in step]

    if missing:
        result["valid"] = False
        result["issues"].append("Missing fields: " + ", ".join(missing))

    return result


print("Loading tokenizer:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


print("Loading model:", MODEL_ID)

if torch.cuda.is_available():
    dtype = torch.bfloat16
    device_map = "auto"
else:
    dtype = torch.float32
    device_map = "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=device_map,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.eval()

print("Model loaded.")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def predict_route(
    target_smiles: str,
    max_new_tokens: int = 1400,
    temperature: float = 0.0,
    top_p: float = 0.9,
    validate: bool = True,
    pretty: bool = True,
    show_raw_on_error: bool = True,
) -> Dict[str, Any]:
    target_can = canonical_smiles(target_smiles)

    if target_can is None:
        raise ValueError(f"Invalid target SMILES: {target_smiles}")

    user_prompt = f"""Target Molecule SMILES: {target_can}
Every route must contain exactly one reaction step.
Every route's only step product_smiles must be this exact target molecule.
Generate exactly 1 distinct route option.
Return valid JSON only."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
    ).to(model.device)

    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=temperature > 0,
            repetition_penalty=1.03,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    elapsed = time.time() - start

    raw_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

    try:
        json_text = extract_json_object(raw_output)
        parsed = json.loads(json_text)
        validation = validate_route_json(parsed, target_can) if validate else None

        result = {
            "ok": True,
            "target_smiles": target_can,
            "seconds": elapsed,
            "validation": validation,
            "result": parsed,
            "raw_output": raw_output,
        }

        if pretty:
            print(json.dumps(parsed, ensure_ascii=False, indent=2))
            print()
            print("Time:", round(elapsed, 2), "sec")
            if validation is not None:
                print("Valid:", validation["valid"])
                if validation["issues"]:
                    print("Issues:", validation["issues"])

        return result

    except Exception as exc:
        result = {
            "ok": False,
            "target_smiles": target_can,
            "seconds": elapsed,
            "error": str(exc),
            "raw_output": raw_output,
        }

        print("JSON parse failed.")
        print("Error:", str(exc))
        print("Time:", round(elapsed, 2), "sec")

        if show_raw_on_error:
            print("\nRAW MODEL OUTPUT:")
            print(raw_output)

        return result

Loading tokenizer: oleh13/ord-retro-qwen25-15b-merged-fp16
Loading model: oleh13/ord-retro-qwen25-15b-merged-fp16


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded.
CUDA: True
GPU: NVIDIA L40S


In [4]:
predict_route("N#CCc1ccccc1")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 1115 (char 1114)
Time: 5.95 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])[O-].[K+].[K+]","N#CCc1ccccc1"],"product_smiles":"N#CCc1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv","reagents":"not specified","solvent":"water","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The crude product was purified by preparative HPLC","expected_yield_percent":"not specified","important_conditions":"CUSTOM: The product was obtained as a solid","rationale":"The proposed single reaction is selected because it directly matches the required one-step format with the evidence reaction ID","objective_fit":"This route directly fits the requirement of a single reaction 

{'ok': False,
 'target_smiles': 'N#CCc1ccccc1',
 'seconds': 5.946622848510742,
 'error': "Expecting ',' delimiter: line 1 column 1115 (char 1114)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])[O-].[K+].[K+]","N#CCc1ccccc1"],"product_smiles":"N#CCc1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv","reagents":"not specified","solvent":"water","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The crude product was purified by preparative HPLC","expected_yield_percent":"not specified","important_conditions":"CUSTOM: The product was obtained as a solid","rationale":"The proposed single reaction is selected because it directly matches the required one-step format with the evidence reaction ID","objective_fit":"This route dir

In [5]:
predict_route("O=[N+]([O-])c1ccccc1")

JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 1027 (char 1026)
Time: 5.22 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=[N+]([O-])c1ccccc1"],"product_smiles":"O=[N+]([O-])c1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"This route is selected because it directly matches the required one-step format with the necessary reaction details derived from the ORD evidence record.","evidence_reaction_ids":["ord-9b9f9d9a9b9f4f079f9f9f9f9f9f9f9f"]}],"overall_recommendation":"This ORD-derived route is re

{'ok': False,
 'target_smiles': 'O=[N+]([O-])c1ccccc1',
 'seconds': 5.215687274932861,
 'error': "Expecting ',' delimiter: line 1 column 1027 (char 1026)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=[N+]([O-])c1ccccc1"],"product_smiles":"O=[N+]([O-])c1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"This route is selected because it directly matches the required one-step format with the necessary reaction details derived from the ORD evidence record.","evidence_reaction_ids":["ord-9b9f9d9a9b9f4f079f9f9f9f9f9f9f9f"]}

In [6]:
predict_route("CC(C)(O)c1ccccc1")

JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 844 (char 843)
Time: 4.48 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["CC(C)(O)c1ccccc1"],"product_smiles":"CC(C)(O)c1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f9c7d9a0f4f9b9f9f9f9f9f9f9f9f"]}],"overall_recommendation":"Single-step ORD-derived route is recommended because it directly matches the required one-step format and uses evidence reaction IDs."}


{'ok': False,
 'target_smiles': 'CC(C)(O)c1ccccc1',
 'seconds': 4.478452205657959,
 'error': "Expecting ',' delimiter: line 1 column 844 (char 843)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["CC(C)(O)c1ccccc1"],"product_smiles":"CC(C)(O)c1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f9c7d9a0f4f9b9f9f9f9f9f9f9f9f"]}],"overall_recommendation":"Single-step ORD-derived route is recommended because it directly matches the required one-step format and uses evidence reacti

In [7]:
predict_route("c1ccc(-c2ccccc2)cc1")

JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 1400 (char 1399)
Time: 7.79 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])[O-].[K+].[K+]","O","O=C(O)c1ccccc1","[Cl-].[NH4+]"],"product_smiles":"c1ccc(-c2ccccc2)cc1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv, Reactant 3: 1.0 equiv, Reactant 4: 1.0 equiv","reagents":"not specified","solvent":"water; ethyl acetate","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The organic layer was separated; WASH: washed with water (2×10 mL); DRY_WITH_MATERIAL: dried over anhydrous sodium sulfate; FILTRATION: filtered; CONCENTRATION: concentrated under reduced pressure; CUSTOM: The residue was purified by silica gel column chromatography (ethyl acetate/hexane)","ex

{'ok': False,
 'target_smiles': 'c1ccc(-c2ccccc2)cc1',
 'seconds': 7.793167352676392,
 'error': "Expecting ',' delimiter: line 1 column 1400 (char 1399)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])[O-].[K+].[K+]","O","O=C(O)c1ccccc1","[Cl-].[NH4+]"],"product_smiles":"c1ccc(-c2ccccc2)cc1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv, Reactant 3: 1.0 equiv, Reactant 4: 1.0 equiv","reagents":"not specified","solvent":"water; ethyl acetate","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The organic layer was separated; WASH: washed with water (2×10 mL); DRY_WITH_MATERIAL: dried over anhydrous sodium sulfate; FILTRATION: filtered; CONCENTRATION: concentrated under reduced pressure; CUSTOM: The residue was purified by silic

In [8]:
predict_route("O=C(/C=C/c1ccccc1)c1ccccc1")

JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 1396 (char 1395)
Time: 7.68 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])O.[Na+]","O=C(O)/C=C/c1ccccc1","c1ccc(P(c2ccccc2)c2ccccc2)cc1"],"product_smiles":"O=C(/C=C/c1ccccc1)c1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv, Reactant 3: 1.0 equiv","reagents":"not specified","solvent":"THF","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The reaction was quenched by addition of water; EXTRACTION: extracted with ethyl acetate; WASH: The organic layer was washed with brine; DRY_WITH_MATERIAL: dried over sodium sulfate; CONCENTRATION: concentrated under reduced pressure; CUSTOM: The residue was purified by silica gel column chromatography (hexane/ethy

{'ok': False,
 'target_smiles': 'O=C(/C=C/c1ccccc1)c1ccccc1',
 'seconds': 7.682173490524292,
 'error': "Expecting ',' delimiter: line 1 column 1396 (char 1395)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])O.[Na+]","O=C(O)/C=C/c1ccccc1","c1ccc(P(c2ccccc2)c2ccccc2)cc1"],"product_smiles":"O=C(/C=C/c1ccccc1)c1ccccc1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv, Reactant 3: 1.0 equiv","reagents":"not specified","solvent":"THF","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The reaction was quenched by addition of water; EXTRACTION: extracted with ethyl acetate; WASH: The organic layer was washed with brine; DRY_WITH_MATERIAL: dried over sodium sulfate; CONCENTRATION: concentrated under reduced pressure; CUSTOM: The residue

In [9]:
predict_route("C=Cc1ccccc1")

JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 835 (char 834)
Time: 4.36 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=Cc1ccccc1","[H][H]"],"product_smiles":"C=Cc1ccccc1","stoichiometry":"not specified","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f7d0a970f499b9f997f9f9f9f9f9f"]}],"overall_recommendation":"Single-step ORD-derived route is recommended because it directly matches the required one-step format and uses evidence reaction IDs."}


{'ok': False,
 'target_smiles': 'C=Cc1ccccc1',
 'seconds': 4.356117248535156,
 'error': "Expecting ',' delimiter: line 1 column 835 (char 834)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=Cc1ccccc1","[H][H]"],"product_smiles":"C=Cc1ccccc1","stoichiometry":"not specified","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f7d0a970f499b9f997f9f9f9f9f9f"]}],"overall_recommendation":"Single-step ORD-derived route is recommended because it directly matches the required one-step format and uses evidence reaction IDs."}'}

In [10]:
predict_route("C1=CCCCC1")

JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 1438 (char 1437)
Time: 7.21 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])O.[Na+]","[H][H]"],"product_smiles":"C1=CCCCC1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv","reagents":"not specified","solvent":"water","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The reaction was quenched by addition of water; EXTRACTION: extracted with ethyl acetate; WASH: The organic layer was washed with brine; DRY_WITH_MATERIAL: dried over sodium sulfate; CONCENTRATION: concentrated under reduced pressure; CUSTOM: The residue was purified by column chromatography on silica gel (eluent: hexane/ethyl acetate)","expected_yield_percent":"not specified","important_conditio

{'ok': False,
 'target_smiles': 'C1=CCCCC1',
 'seconds': 7.213254451751709,
 'error': "Expecting ',' delimiter: line 1 column 1438 (char 1437)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["O=C([O-])O.[Na+]","[H][H]"],"product_smiles":"C1=CCCCC1","stoichiometry":"Reactant 1: 1.0 equiv, Reactant 2: 1.0 equiv","reagents":"not specified","solvent":"water","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"CUSTOM: The reaction was quenched by addition of water; EXTRACTION: extracted with ethyl acetate; WASH: The organic layer was washed with brine; DRY_WITH_MATERIAL: dried over sodium sulfate; CONCENTRATION: concentrated under reduced pressure; CUSTOM: The residue was purified by column chromatography on silica gel (eluent: hexane/ethyl acetate)","expected_yield

In [11]:
predict_route("CC(O)c1ccccc1")

JSON parse failed.
Error: Expecting ',' delimiter: line 1 column 843 (char 842)
Time: 4.59 sec

RAW MODEL OUTPUT:
{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["CC(=O)C1CC(=O)CC1","[H][H]"],"product_smiles":"CC(O)c1ccccc1","stoichiometry":"not specified","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f7d9c7a9e4f9b9f7f9f9f9f9f9f9f"]}],"overall_recommendation":"Single-step ORD-derived route is recommended because it directly matches the required one-step format and uses evidence reaction IDs."}


{'ok': False,
 'target_smiles': 'CC(O)c1ccccc1',
 'seconds': 4.5880982875823975,
 'error': "Expecting ',' delimiter: line 1 column 843 (char 842)",
 'raw_output': '{"routes":[{"route_name":"Single-step ORD-derived route","summary":"One-step ORD-derived route to produce the target molecule","steps":[{"reaction_name":"ORD-derived reaction","reactants":["CC(=O)C1CC(=O)CC1","[H][H]"],"product_smiles":"CC(O)c1ccccc1","stoichiometry":"not specified","reagents":"not specified","solvent":"not specified","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f7d9c7a9e4f9b9f7f9f9f9f9f9f9f"]}],"overall_recommendation":"Single-step ORD-derived route is recommended because it directly matches the required one-step format and uses evidence reaction 

In [12]:
predict_route("O=C1OC(=O)C2CC=CCC12")

JSON parse failed.
Error: Expecting ',' delimiter: line 12 column 1 (char 950)
Time: 6.0 sec

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Single-step ORD-derived route",
      "summary": "One-step ORD-derived route to the target product",
      "steps": [
        {
          "reaction_name": "ORD-derived reaction",
          "reactants": ["O=C(O)C1CC=CCC1","[Na+].[OH-]","Cl"],"product_smiles": "O=C1OC(=O)C2CC=CCC12","stoichiometry":"not specified","reagents":"not specified","solvent":"ethanol","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f0d9c9a9f4f9b9f0f7f9f9f9f9f9f"]}],
      "objective_fit": "This route is selected because it is a direct ORD-derived single-step proposal for the target product",
      "evid

{'ok': False,
 'target_smiles': 'O=C1OC(=O)C2CC=CCC12',
 'seconds': 5.995607852935791,
 'error': "Expecting ',' delimiter: line 12 column 1 (char 950)",
 'raw_output': '{\n  "routes": [\n    {\n      "route_name": "Single-step ORD-derived route",\n      "summary": "One-step ORD-derived route to the target product",\n      "steps": [\n        {\n          "reaction_name": "ORD-derived reaction",\n          "reactants": ["O=C(O)C1CC=CCC1","[Na+].[OH-]","Cl"],"product_smiles": "O=C1OC(=O)C2CC=CCC12","stoichiometry":"not specified","reagents":"not specified","solvent":"ethanol","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"not specified","expected_yield_percent":"not specified","important_conditions":"not specified","rationale":"not specified","objective_fit":"not specified","evidence_reaction_ids":["ord-9b9f0d9c9a9f4f9b9f0f7f9f9f9f9f9f"]}],\n      "objective_fit": "This route is selected because it is a direct ORD

In [13]:
predict_route("CCNC(C)=O")

JSON parse failed.
Error: Expecting ',' delimiter: line 9 column 963 (char 1184)
Time: 5.92 sec

RAW MODEL OUTPUT:
{
  "routes": [
    {
      "route_name": "Single-step ORD-derived route",
      "summary": "One-step ORD-derived route to the target product",
      "steps": [
        {
          "reaction_name": "ORD-derived reaction",
          "reactants": ["CCN=C=O","[H][H]"],"reagents":"Pd/C","solvent":"EtOH","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"FILTRATION: The catalyst was filtered off; CONCENTRATION: the filtrate was concentrated under reduced pressure; CUSTOM: The residue was purified by column chromatography (silica gel, EtOAc/Hexane)","expected_yield_percent":"not specified","important_conditions":"See reaction.notes.procedure_details.","rationale":"The listed reaction conditions are based on the ORD evidence reaction ID.","objective_fit":"This route is selected because it directly matches the 

{'ok': False,
 'target_smiles': 'CCNC(C)=O',
 'seconds': 5.924162864685059,
 'error': "Expecting ',' delimiter: line 9 column 963 (char 1184)",
 'raw_output': '{\n  "routes": [\n    {\n      "route_name": "Single-step ORD-derived route",\n      "summary": "One-step ORD-derived route to the target product",\n      "steps": [\n        {\n          "reaction_name": "ORD-derived reaction",\n          "reactants": ["CCN=C=O","[H][H]"],"reagents":"Pd/C","solvent":"EtOH","temperature_celsius":"not specified","reaction_time":"not specified","atmosphere":"not specified","workup_purification":"FILTRATION: The catalyst was filtered off; CONCENTRATION: the filtrate was concentrated under reduced pressure; CUSTOM: The residue was purified by column chromatography (silica gel, EtOAc/Hexane)","expected_yield_percent":"not specified","important_conditions":"See reaction.notes.procedure_details.","rationale":"The listed reaction conditions are based on the ORD evidence reaction ID.","objective_fit":"Th